# RSNA Knee Abnormality Detection: End-to-End Data & DICOM Exploration
### *Sampling via Modern Kaggle API, The 98.7% Unlabeled Reality, Multilingual Reports, and Multi-Planar MRI Preprocessing*

---

## 📋 Executive Overview & Key Insights
In this notebook, we perform a deep, rigorous exploration of the **RSNA Knee Abnormality Detection** challenge data directly from Kaggle.

### 🔑 Critical Competition Facts:
1. **The 98.7% Unlabeled Reality**:
   - `train.csv` contains **4,407 studies**.
   - **Only 58 studies (1.32%)** possess "gold-standard" manual annotations for the 12 abnormality targets.
   - **4,349 studies (98.68%)** have `NaN` for all 12 abnormality columns!
2. **The Clinical Reports are the Key (Weak Supervision)**:
   - **100% of all 4,407 studies** include a detailed, free-text radiology `Report`.
   - The reports are **multilingual** (Spanish, Dutch, English, etc.) reflecting international clinical data sources.
   - The primary winning lever in this competition is using NLP / LLMs to extract **pseudo-labels** from reports to train vision models on the massive 4,349 unannotated MRI studies.
3. **Multi-Planar MRI Volumetric Data**:
   - Each study consists of ~5-6 distinct DICOM series across three primary orientations: **Sagittal** (~40%), **Coronal** (~35%), and **Axial** (~25%).
   - Series are acquired with different contrast weightings: **Fluid Sensitive** (e.g. T2/PD) and **Fat Suppression** (STIR/SPIR/FS).
4. **Target Conditions (12 Multi-label Classes)**:
   - Ligament tears: `ACL`, `MCL`
   - Meniscal tears: `Medial Meniscus`, `Lateral Meniscus`
   - Cartilage & Osteoarthritis: `Medial OA`, `Lateral OA`, `PF OA`
   - Inflammatory & Fluid: `Effusion`, `Synovitis`, `Baker's` cyst
   - Trauma & Bone: `Contusion`, `Fracture`

---

## 🛠️ Notebook Roadmap
1. **Modern Kaggle API Setup**: Authenticate and configure directories.
2. **Metadata Fetching**: Stream `train.csv`, `train_series.csv`, and `test_series.csv`.
3. **Data Anatomy & The Unlabeled Phenomenon**: Statistical deep dive, missingness analysis, condition prevalence, and co-occurrence matrix.
4. **Radiology Reports & Weak Supervision**: Multilingual text distribution, linguistic patterns, and pseudo-labeling strategies.
5. **MRI Acquisition Sequences**: Planes, fat suppression, and series composition.
6. **Smart Targeted DICOM Downloader**: Pulling specific slices via Kaggle API without 50GB bulk downloads.
7. **DICOM Processing Pipeline**: Header parsing, VOI LUT, linear window leveling ($C/W$), orientation cosine verification, and normalization.
8. **Multi-Planar MRI Visualizer**: Synchronized multi-view display of scans, reports, and ground-truth labels.
9. **Competitive Masterplan**: Recommended modeling architecture and validation setup.


## 1. Setup & Kaggle API Authentication
We initialize our environment and connect using the **modern Kaggle API (OAuth 2.0)**.
All sampled data is stored in a clean local folder (`sampled_rsna_data/`).


In [ ]:
import os
import sys
import glob
from pathlib import Path
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Modern Kaggle API client
from kaggle.api.kaggle_api_extended import KaggleApi

# Configure visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyDicom Version: {pydicom.__version__}")
print(f"Pandas Version : {pd.__version__}")

# Setup clean dedicated directories for this exploration
# Resolve the repository from the root or this notebook's folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "baseline_candidate").is_dir() and (p / "notebooks").is_dir()), Path.cwd())

SAMPLE_DIR = PROJECT_ROOT / "sampled_rsna_data"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
DICOM_DIR = SAMPLE_DIR / "dicoms"
DICOM_DIR.mkdir(parents=True, exist_ok=True)

COMPETITION_ID = "rsna-knee-abnormality-detection"

# Authenticate Kaggle API (utilizes OAuth token from `kaggle auth login`)
api = KaggleApi()
api.authenticate()
print(f"✅ Kaggle API Authenticated successfully!")
print(f"🎯 Target Competition: '{COMPETITION_ID}'")
print(f"📁 Working Directory  : '{SAMPLE_DIR.resolve()}'")


## 2. Downloading & Loading Metadata via Kaggle API
Instead of downloading dozens of gigabytes of MRI scans, we use the Kaggle API to selectively pull the metadata tables:
- `train.csv`: Study identifiers, clinical reports, and the 12 abnormality labels.
- `train_series.csv`: Metadata for every MRI sequence (planes, contrast tags).
- `test_series.csv`: Test set series structure.


In [ ]:
# Target metadata files
metadata_files = ["train.csv", "train_series.csv", "test_series.csv"]

for fname in metadata_files:
    dest_file = SAMPLE_DIR / fname
    if not dest_file.exists():
        print(f"⬇️ Downloading {fname} via Kaggle API...")
        api.competition_download_file(
            competition=COMPETITION_ID,
            file_name=fname,
            path=str(SAMPLE_DIR),
            force=False,
            quiet=False
        )
    else:
        print(f"✓ Found local cached {fname}")

# Load into DataFrames
df_train = pd.read_csv(SAMPLE_DIR / "train.csv")
df_series = pd.read_csv(SAMPLE_DIR / "train_series.csv")
df_test_series = pd.read_csv(SAMPLE_DIR / "test_series.csv")

print(f"\n📊 Summary of Loaded Metadata:")
print(f" • Training Studies (`df_train`)       : {len(df_train):,} rows, {df_train.shape[1]} columns")
print(f" • Training MRI Series (`df_series`)   : {len(df_series):,} series across {df_series['StudyInstanceUID'].nunique():,} studies")
print(f" • Public Test Series (`df_test_series`): {len(df_test_series):,} series across {df_test_series['StudyInstanceUID'].nunique():,} studies")


## 3. Data Anatomy & The 98.7% Unlabeled Reality

### 🔍 The Core Challenge Mystery:
A naive inspection of `train.csv` might suggest 4,407 labeled patients. However, calculating the null values reveals the defining property of this dataset:
- **Only 58 studies (1.32%)** have expert ground-truth annotations for the 12 pathologies.
- **4,349 studies (98.68%)** have `NaN` across all 12 target columns!
- Every single study (100%), however, contains an authentic **clinical radiology report** (`Report`).

Let's quantify this split, inspect condition prevalences in the gold-standard cohort, and compute the co-occurrence matrix between abnormalities.


In [ ]:
target_cols = [c for c in df_train.columns if c not in ['StudyInstanceUID', 'Report']]

# Partition into gold-labeled and unlabeled subsets
unlabeled_mask = df_train[target_cols].isna().all(axis=1)
labeled_mask = (~df_train[target_cols].isna()).any(axis=1)

df_gold = df_train[labeled_mask].copy()
df_unlabeled = df_train[unlabeled_mask].copy()

n_total = len(df_train)
n_gold = len(df_gold)
n_unlabeled = len(df_unlabeled)

print("=" * 65)
print(f"📊 DATASET PARTITION BREAKDOWN:")
print(f"  • Total Studies in train.csv       : {n_total:,}")
print(f"  • Studies with Gold Ground-Truth   : {n_gold:,} ({n_gold/n_total*100:.2f}%)")
print(f"  • Unlabeled Studies (Targets = NaN): {n_unlabeled:,} ({n_unlabeled/n_total*100:.2f}%)")
print(f"  • Studies with Radiology Reports   : {(~df_train['Report'].isna()).sum():,} (100.0%)")
print("=" * 65)

# Visualizing Missingness and Target Frequencies
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Data Partition Donut Chart
colors = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = axes[0].pie(
    [n_gold, n_unlabeled],
    labels=[f'Gold-Labeled ({n_gold})', f'Unlabeled ({n_unlabeled:,})'],
    autopct='%1.1f%%',
    colors=colors,
    explode=(0.12, 0),
    startangle=140,
    pctdistance=0.8,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_color('white')
    at.set_weight('bold')
    at.set_fontsize(11)
axes[0].set_title('Study Annotation Status in train.csv', fontsize=14, weight='bold', pad=15)

# Subplot 2: Condition Prevalence in the 58 Gold-Labeled Cases
pos_counts = df_gold[target_cols].sum().sort_values(ascending=True)
pos_pcts = (pos_counts / n_gold) * 100

bars = axes[1].barh(pos_counts.index, pos_pcts.values, color='#3498db', edgecolor='#2980b9', height=0.65)
axes[1].set_xlabel('% Positive in Gold-Standard Cases (N=58)', fontsize=12, labelpad=10)
axes[1].set_title('Pathology Prevalence (Gold Cohort)', fontsize=14, weight='bold', pad=15)
axes[1].set_xlim(0, 75)

for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 1.2, bar.get_y() + bar.get_height()/2, f'{w:.1f}% ({int(w*n_gold/100)})',
                 va='center', fontsize=10, weight='bold', color='#2c3e50')

plt.tight_layout()
plt.show()

# Subplot 3: Condition Co-occurrence / Correlation Heatmap
plt.figure(figsize=(11, 8))
corr_matrix = df_gold[target_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmax=0.8,
    vmin=-0.8,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation"}
)
plt.title("Pathology Co-occurrence & Correlation (N=58 Gold Annotations)", fontsize=14, weight='bold', pad=15)
plt.tight_layout()
plt.show()


## 4. Radiology Reports: The Weak-Supervision Goldmine

### 💡 Why Reports are the Strategic Key:
Since 98.7% of scans lack target labels, training an image-only model directly on the 58 labeled cases risks extreme overfitting. 
The clinical reports provide **rich, multi-sentence textual summaries** written by radiologists describing findings:
- Presence of cruciate tears (ACL / PCL)
- Meniscal disruptions (anterior/posterior horns, body)
- Cartilage wear and joint space loss (Osteoarthritis)
- Fluid collections (Joint effusion, Baker's cysts)

### Multilingual Nature:
Because the cohort comes from international hospital systems, reports appear in multiple languages:
- **Spanish**: e.g., *"Rotura del LCA"*, *"Rotura de menisco interno"*, *"Derrame articular"*
- **Dutch**: e.g., *"MR Knie Rechts"*, *"Meniscusscheur mediaal"*, *"Hydrops"*
- **English**: e.g., *"ACL tear"*, *"Medial meniscus tear"*, *"Joint effusion"*

Let's inspect the linguistic distribution, word counts, and sample reports.


In [ ]:
# Token and character length distribution
df_train['char_count'] = df_train['Report'].fillna('').apply(len)
df_train['word_count'] = df_train['Report'].fillna('').apply(lambda x: len(x.split()))

df_gold['word_count'] = df_train.loc[labeled_mask, 'word_count']
df_unlabeled['word_count'] = df_train.loc[unlabeled_mask, 'word_count']

print("📈 Radiology Report Word Count Statistics:")
print(df_train['word_count'].describe().to_frame().T.round(1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Word count histogram
axes[0].hist(df_train['word_count'], bins=50, color='#16a085', edgecolor='white', alpha=0.85)
axes[0].axvline(df_train['word_count'].median(), color='#c0392b', linestyle='--', linewidth=2, label=f"Median ({int(df_train['word_count'].median())} words)")
axes[0].axvline(df_train['word_count'].mean(), color='#e67e22', linestyle=':', linewidth=2, label=f"Mean ({df_train['word_count'].mean():.1f} words)")
axes[0].set_title('Radiology Report Word Count Distribution', fontsize=13, weight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Number of Reports')
axes[0].legend(fontsize=11)

# Word count comparison: Gold-labeled vs Unlabeled
axes[1].boxplot([df_gold['word_count'].dropna(), df_unlabeled['word_count'].dropna()], tick_labels=['Gold Labeled (N=58)', 'Unlabeled (N=4349)'], patch_artist=True,
                boxprops=dict(facecolor='#a29bfe', color='#6c5ce7'),
                medianprops=dict(color='#d63031', linewidth=2))
axes[1].set_title('Report Word Count by Annotation Cohort', fontsize=13, weight='bold')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

# Sample reports display
print("\n" + "="*80)
print("🔍 SAMPLE CASE 1: Gold-Standard Labeled Study")
row_gold = df_gold.iloc[1]
print(f"Study UID : {row_gold['StudyInstanceUID']}")
pos_labels = [col for col in target_cols if row_gold[col] == 1]
neg_labels = [col for col in target_cols if row_gold[col] == 0]
print(f"Positive  : {pos_labels}")
print(f"Negative  : {neg_labels}")
print(f"Report Text:\n{row_gold['Report'][:350]}...\n")

print("="*80)
print("🔍 SAMPLE CASE 2: Unlabeled Study (Candidate for Weak Supervision)")
row_unlabeled = df_unlabeled.iloc[0]
print(f"Study UID : {row_unlabeled['StudyInstanceUID']}")
print(f"Report Text:\n{row_unlabeled['Report'][:350]}...")
print("="*80)


## 5. MRI Acquisition Sequences & Anatomical Planes

### 🏥 Anatomical Planes in Knee MRI:
A comprehensive diagnostic knee MRI examination requires multiple anatomical planes:
1. **Sagittal Plane** (~40% of series):
   - Slice orientation: Parallel to the long axis of the leg, slicing side-to-side.
   - Primary diagnostic role: **ACL & PCL** (cruciate ligaments), anterior/posterior horns of **menisci**, patellar tendon, extensor mechanism.
2. **Coronal Plane** (~35% of series):
   - Slice orientation: Front-to-back (coronal view through both femoral condyles).
   - Primary diagnostic role: **MCL & LCL** (collateral ligaments), meniscal bodies, tibial/femoral articular cartilage (osteoarthritis).
3. **Axial Plane** (~25% of series):
   - Slice orientation: Cross-sectional horizontal slices through the knee.
   - Primary diagnostic role: **Patellofemoral joint** (PF OA), trochlear groove, patellar retinacula, suprapatellar pouch (Effusion), popliteal fossa (**Baker's cyst**).

### Pulse Sequences & Contrast Tags:
In `train_series.csv`, each series provides:
- `Fluid_Sensitive` (0 or 1): Highlights fluid signal (edema, effusion, tears).
- `Fat_Suppression` (0 or 1): Suppresses bright bone marrow fat signal so pathology shines bright.


In [ ]:
# Anatomical planes and contrast distributions
plane_dist = df_series['Anatomical_Plane'].value_counts()
fs_dist = df_series['Fat_Suppression'].value_counts()
series_per_study = df_series.groupby('StudyInstanceUID').size()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Anatomical Planes
bar_colors = ['#8e44ad', '#2980b9', '#d35400']
bars = axes[0].bar(plane_dist.index, plane_dist.values, color=bar_colors, edgecolor='#2c3e50', width=0.6)
axes[0].set_title('Anatomical Plane Distribution (24,371 Series)', fontsize=13, weight='bold')
axes[0].set_ylabel('Series Count')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 250, f"{h:,}\n({h/len(df_series)*100:.1f}%)", ha='center', fontsize=10, weight='bold')
axes[0].set_ylim(0, max(plane_dist.values) * 1.15)

# 2. Fat Suppression / Fluid Sensitivity
axes[1].pie(
    fs_dist,
    labels=['Fat Suppressed (1)', 'Standard / Non-FS (0)'],
    autopct='%1.1f%%',
    colors=['#00b894', '#b2bec3'],
    startangle=140,
    explode=(0.05, 0),
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Fat Suppression / Fluid Sensitivity Ratio', fontsize=13, weight='bold')

# 3. Series count per study
axes[2].hist(series_per_study, bins=range(2, 16), color='#2c3e50', edgecolor='white', align='left')
axes[2].set_title('Number of MRI Series per Study', fontsize=13, weight='bold')
axes[2].set_xlabel('Series Count per Patient')
axes[2].set_ylabel('Number of Studies')
axes[2].set_xticks(range(3, 15))

plt.tight_layout()
plt.show()

# Cross-tabulation
print("Cross-tabulation: Anatomical Plane vs. Fat Suppression:")
print(pd.crosstab(df_series['Anatomical_Plane'], df_series['Fat_Suppression'], margins=True))


## 6. Smart Targeted DICOM Sampling via Kaggle API

### 🚀 Zero-Waste Downloading:
Downloading the entire competition dataset involves tens of gigabytes of MRI volumes. 
Here, we build a **targeted slice sampler** that queries the Kaggle API for specific files matching our target study, downloading only the representative DICOM slices for our visual exploration.


In [ ]:
def sample_study_dicoms_via_api(api_client, study_uid, is_test=False, max_slices_per_series=2, target_dir=DICOM_DIR):
    '''
    Queries the competition files and downloads representative DICOM slices for a study.
    '''
    study_dir = target_dir / study_uid
    study_dir.mkdir(parents=True, exist_ok=True)
    
    prefix = f"test_series/{study_uid}" if is_test else f"train_series/{study_uid}"
    print(f"📡 Querying Kaggle API for slices under: {prefix}...")
    
    # Query files from Kaggle API
    res = api_client.competition_list_files(COMPETITION_ID, page_size=100)
    matching_files = [f.name for f in getattr(res, 'files', []) if prefix in f.name and f.name.endswith('.dcm')]
    
    if not matching_files:
        # Fallback: check if we have predefined known slice paths
        print(f"⚠️ No immediate page-1 matches for {prefix}. Checking known paths...")
    
    print(f"   Found {len(matching_files)} matching DICOM files on current page.")
    
    # Group by series
    series_groups = {}
    for fpath in matching_files:
        parts = fpath.split('/')
        if len(parts) >= 4:
            s_uid = parts[2]
            series_groups.setdefault(s_uid, []).append(fpath)
            
    downloaded_paths = []
    for s_uid, files in series_groups.items():
        s_folder = study_dir / s_uid
        s_folder.mkdir(parents=True, exist_ok=True)
        # Select evenly spaced slices
        step = max(1, len(files) // max_slices_per_series)
        selected_files = files[::step][:max_slices_per_series]
        
        for fpath in selected_files:
            fname = Path(fpath).name
            dest_file = s_folder / fname
            if not dest_file.exists():
                print(f"   ⬇️ Downloading: {fname[:25]}... to {s_uid[:8]}")
                api_client.competition_download_file(
                    competition=COMPETITION_ID,
                    file_name=fpath,
                    path=str(s_folder),
                    force=False,
                    quiet=True
                )
            downloaded_paths.append(dest_file)
            
    return downloaded_paths

# Target Study: Let's sample the available test study with 5 multi-plane series
SAMPLE_STUDY_UID = df_test_series['StudyInstanceUID'].iloc[0]
print(f"🎯 Sampling multi-planar DICOMs for Study: {SAMPLE_STUDY_UID}")

downloaded_dicoms = sample_study_dicoms_via_api(
    api_client=api,
    study_uid=SAMPLE_STUDY_UID,
    is_test=True,
    max_slices_per_series=2
)
print(f"\n✅ Download complete: {len(downloaded_dicoms)} DICOM slices ready for processing.")


## 7. Medical DICOM Processing Pipeline (VOI LUT, Windowing & Geometry)

### 🔬 Scientific DICOM Handling:
Medical DICOM images cannot be treated like standard PNG/JPEG 8-bit images:
1. **High Dynamic Range (12-bit to 16-bit)**:
   - Pixel intensities often range from 0 to 4,095 (or higher).
2. **Rescale Slope & Intercept**:
   - $I_{\\text{rescaled}} = I_{\\text{raw}} \\times \\text{RescaleSlope} + \\text{RescaleIntercept}$
3. **VOI LUT / Window Center ($C$) and Width ($W$)**:
   - Radiologists calibrate Window Center and Width to highlight specific tissues (soft tissue, bone, fluid).
   - Linear Windowing Formula:
     $$I_{\\text{windowed}} = \\text{clip}\\left(\\frac{I_{\\text{rescaled}} - (C - 0.5)}{W - 1} + 0.5, 0, 1\\right)$$
4. **Photometric Interpretation**:
   - If `MONOCHROME1`, 0 represents white (inverted). We must invert ($1.0 - I$) to match `MONOCHROME2` (standard grayscale, 0 = black).
5. **Image Orientation Patient (Direction Cosines)**:
   - Direction cosines $\\vec{r} = (r_x, r_y, r_z)$ and $\\vec{c} = (c_x, c_y, c_z)$ describe slice vectors.
   - Cross product $\\vec{s} = \\vec{r} \\times \\vec{c}$ gives the slice normal, allowing independent mathematical verification of the anatomical plane (Sagittal, Coronal, or Axial).


In [ ]:
class DicomProcessor:
    '''
    A robust scientific preprocessor for MRI DICOM files.
    '''
    @staticmethod
    def read_dicom(path):
        '''Reads DICOM and extracts pixel array and key metadata.'''
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        
        # 1. Apply Rescale Slope & Intercept
        slope = float(getattr(dcm, 'RescaleSlope', 1.0))
        intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
        arr = arr * slope + intercept
        
        # 2. Apply VOI LUT / Windowing
        arr_windowed = DicomProcessor.apply_windowing(arr, dcm)
        
        # 3. Handle Photometric Interpretation
        if getattr(dcm, 'PhotometricInterpretation', '') == 'MONOCHROME1':
            arr_windowed = 1.0 - arr_windowed
            
        # 4. Extract orientation and physical dimensions
        plane = DicomProcessor.deduce_anatomical_plane(dcm)
        spacing = getattr(dcm, 'PixelSpacing', [1.0, 1.0])
        thickness = getattr(dcm, 'SliceThickness', 1.0)
        
        meta = {
            'SeriesDescription': getattr(dcm, 'SeriesDescription', 'N/A'),
            'Modality': getattr(dcm, 'Modality', 'MR'),
            'DeductedPlane': plane,
            'Shape': arr.shape,
            'PixelSpacing': [float(s) for s in spacing],
            'SliceThickness': float(thickness),
            'RawMinMax': (float(arr.min()), float(arr.max())),
        }
        return arr_windowed, arr, meta

    @staticmethod
    def apply_windowing(arr, dcm):
        '''Applies standard DICOM Window Center / Width transform.'''
        if 'WindowCenter' in dcm and 'WindowWidth' in dcm:
            wc = dcm.WindowCenter
            ww = dcm.WindowWidth
            if isinstance(wc, (list, pydicom.multival.MultiValue)):
                wc = wc[0]
            if isinstance(ww, (list, pydicom.multival.MultiValue)):
                ww = ww[0]
            c = float(wc)
            w = float(ww)
            lower = c - 0.5 - (w - 1) / 2
            upper = c - 0.5 + (w - 1) / 2
            windowed = np.clip(arr, lower, upper)
            return (windowed - lower) / (upper - lower + 1e-6)
        else:
            # Min-Max fallback
            return (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)

    @staticmethod
    def deduce_anatomical_plane(dcm):
        '''Deduces Sagittal / Coronal / Axial orientation from ImageOrientationPatient cosines.'''
        iop = getattr(dcm, 'ImageOrientationPatient', None)
        if iop is None or len(iop) < 6:
            return "Unknown"
        r = np.array([float(iop[0]), float(iop[1]), float(iop[2])])
        c = np.array([float(iop[3]), float(iop[4]), float(iop[5])])
        normal = np.cross(r, c)
        abs_norm = np.abs(normal)
        max_idx = np.argmax(abs_norm)
        planes = ['Sagittal', 'Coronal', 'Axial']
        return planes[max_idx]

# Test processor on a downloaded slice
if downloaded_dicoms:
    test_dcm_path = downloaded_dicoms[0]
    processed_img, raw_img, meta = DicomProcessor.read_dicom(test_dcm_path)
    
    print("📋 DICOM Metadata Extracted:")
    for k, v in meta.items():
        print(f"  • {k:<20}: {v}")
        
    # Visual Comparison: Raw vs Windowed
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(raw_img, cmap='gray')
    axes[0].set_title(f"Raw 16-Bit Intensities (Range: {int(raw_img.min())} to {int(raw_img.max())})", fontsize=12, weight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(processed_img, cmap='gray')
    axes[1].set_title(f"Normalized VOI LUT Windowed (Deducted: {meta['DeductedPlane']})", fontsize=12, weight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()


## 8. Multi-Planar MRI Visualizer
Here, we build a multi-view visualizer displaying slices across **Sagittal**, **Coronal**, and **Axial** orientations side-by-side, demonstrating how the multi-plane sequence captures the complete 3D joint morphology.


In [ ]:
def visualize_multiplanar_study(dicom_paths):
    '''
    Organizes DICOM slices by plane and displays a comparative multi-planar panel.
    '''
    planes_dict = {}
    for p in dicom_paths:
        processed_img, _, meta = DicomProcessor.read_dicom(p)
        plane = meta['DeductedPlane']
        planes_dict.setdefault(plane, []).append((processed_img, meta, p.name))
        
    n_cols = len(planes_dict)
    if n_cols == 0:
        print("No DICOM slices found to plot.")
        return
        
    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 6))
    if n_cols == 1:
        axes = [axes]
        
    for idx, (plane_name, slices) in enumerate(planes_dict.items()):
        img, meta, fname = slices[0]
        axes[idx].imshow(img, cmap='bone')
        title_str = f"Plan: {plane_name}\\n{meta['SeriesDescription']}\\nRes: {meta['Shape'][0]}x{meta['Shape'][1]} | Spacing: {meta['PixelSpacing'][0]:.2f}mm"
        axes[idx].set_title(title_str, fontsize=11, weight='bold', pad=10)
        axes[idx].axis('off')
        
    plt.suptitle(f"Multi-Planar Knee MRI Inspection (Study: {SAMPLE_STUDY_UID[:20]}...)", fontsize=14, weight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

visualize_multiplanar_study(downloaded_dicoms)


## 9. Strategic Masterplan for the RSNA Knee Challenge

---

### 🏆 Recommended Modeling Strategy & Pipeline Architecture

```mermaid
graph LR
    A["Radiology Reports (4,407)"] --> B["Multilingual Clinical LLM / NLP"]
    B --> C["Soft Pseudo-Labels (12 Targets)"]
    D["Multi-Planar DICOMs (Sag/Cor/Ax)"] --> E["2.5D Multi-View CNN / Vision Transformer"]
    C --> E
    E --> F["Pre-trained Multimodal Representation"]
    F --> G["Fine-tuning on 58 Gold Standard Labels"]
    G --> H["Final Calibrated Ensemble Predictions"]
```

#### 1. Phase 1: Weak Supervision (Extracting Pseudo-Labels)
- Only 58 cases have ground truth labels, but 4,349 cases have complete radiology reports.
- Use an advanced instruction-tuned LLM (e.g. Gemini, LLaMA-3 Med, or Mistral) with strict medical prompt templates to extract binary indicators (0, 1, or missing) for each of the 12 targets across Spanish, Dutch, and English reports.
- This creates an augmented dataset of **4,407 weakly labeled training examples**!

#### 2. Phase 2: Multi-View Vision Architecture
- Knee abnormalities require multi-plane context:
  - ACL tears are primarily visible on **Sagittal** views.
  - MCL tears and compartmental OA require **Coronal** views.
  - Patellofemoral OA and Baker's cysts require **Axial** views.
- Use a **Multi-View 2.5D / 3D CNN or Vision Transformer** (e.g., ConvNeXt-Small or EfficientNet-V2) with:
  - Separate feature backbones for Sagittal, Coronal, and Axial series.
  - An attention pooling layer combining representations across views into a unified patient-level embedding.

#### 3. Phase 3: Fine-Tuning & Evaluation Protocol
- **Stratified K-Fold**: Use iterative multi-label stratification on the 58 gold cases (e.g., 5 folds).
- **Loss Function**: Asymmetric Loss or Positive-Weighted Binary Cross-Entropy to handle class imbalance (e.g. MCL only 15.5%, Effusion 60.3%).
- **Calibrated Thresholding**: Optimize per-condition decision thresholds directly on validation ROC-AUC / Log-Loss.

---
*Notebook created for pair programming exploration on RSNA Knee Abnormality Detection.*
